# Notebook 01 – Chạy Thử PoC trên Dataset Mẫu

**Mục tiêu:** Load dataset thực từ CSV, khởi tạo RL orchestrator (Q-Learning), huấn luyện nhanh, và chạy inference thử.

**Pipeline:**
```
dataset_orchestrator.csv
       ↓
Feature Extraction (keyword relevance → discrete state)
       ↓
Q-Learning Orchestrator (ε-greedy, 27 states × 3 actions)
       ↓
Agent Selection → Reward → Q-table update
       ↓
Inference trên queries mới
```

## 0. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import random
import json
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

DATA_DIR = '../data/'
print('Setup xong.')

## 1. Load Dataset Orchestrator từ CSV

In [ ]:
df_orch = pd.read_csv(DATA_DIR + 'dataset_orchestrator.csv')
print(f'Shape: {df_orch.shape}')
print(f'Columns: {list(df_orch.columns)}')
df_orch.head(10)

In [ ]:
print('Phân bố agent:')
print(df_orch['correct_agent'].value_counts())
print()
print('Phân bố difficulty:')
print(df_orch['difficulty'].value_counts())
print()
print('Has attachment:')
print(df_orch['has_attachment'].value_counts())

In [ ]:
# Lấy mẫu 30 query để demo nhanh (10 per agent)
df_sample = df_orch.groupby('correct_agent', group_keys=False).apply(
    lambda x: x.sample(min(10, len(x)), random_state=42)
).reset_index(drop=True)

print(f'Sample size: {len(df_sample)} queries')
print(df_sample['correct_agent'].value_counts())
df_sample[['query', 'correct_agent', 'difficulty']].head()

## 2. Định nghĩa Agents

In [ ]:
@dataclass
class AgentResult:
    agent_name: str
    confidence: float
    result: dict
    execution_time_ms: float


class FundamentalAgent:
    name = 'fundamental'
    keywords = [
        'P/E', 'PE', 'ROE', 'EPS', 'doanh thu', 'lợi nhuận', 'revenue',
        'báo cáo tài chính', 'financial report', 'cổ tức', 'dividend',
        'giá trị nội tại', 'intrinsic value', 'book value', 'P/B',
        'biên lợi nhuận', 'margin', 'nợ', 'debt', 'equity',
        'balance sheet', 'income statement', 'cash flow',
        'định giá', 'valuation', 'DCF', 'fundamental'
    ]

    def execute(self, query: str, context: dict = None) -> AgentResult:
        relevance = self._calculate_relevance(query)
        result = {
            'analysis_type': 'fundamental',
            'metrics': {
                'P/E': round(random.uniform(8, 35), 2),
                'ROE': round(random.uniform(0.05, 0.30), 4),
                'EPS': round(random.uniform(1000, 15000), 0),
                'Debt_to_Equity': round(random.uniform(0.1, 2.5), 2),
            },
            'recommendation': random.choice(['BUY', 'HOLD', 'SELL']),
            'relevance_score': relevance
        }
        return AgentResult(
            agent_name=self.name, confidence=relevance,
            result=result, execution_time_ms=random.uniform(100, 500)
        )

    def _calculate_relevance(self, query: str) -> float:
        query_lower = query.lower()
        matches = sum(1 for kw in self.keywords if kw.lower() in query_lower)
        return min(1.0, matches * 0.25 + 0.1)


class TechnicalAgent:
    name = 'technical'
    keywords = [
        'MA', 'SMA', 'EMA', 'RSI', 'MACD', 'Bollinger', 'xu hướng',
        'trend', 'support', 'resistance', 'hỗ trợ', 'kháng cự',
        'nến', 'candlestick', 'chart', 'biểu đồ', 'volume',
        'khối lượng', 'breakout', 'momentum', 'stochastic',
        'đường trung bình', 'moving average', 'tín hiệu',
        'signal', 'mua bán', 'entry', 'exit', 'technical'
    ]

    def execute(self, query: str, context: dict = None) -> AgentResult:
        relevance = self._calculate_relevance(query)
        result = {
            'analysis_type': 'technical',
            'indicators': {
                'RSI_14': round(random.uniform(20, 80), 2),
                'MACD_signal': random.choice(['BULLISH', 'BEARISH', 'NEUTRAL']),
                'SMA_50_vs_200': random.choice(['GOLDEN_CROSS', 'DEATH_CROSS', 'NEUTRAL']),
                'Support': round(random.uniform(20000, 50000), 0),
                'Resistance': round(random.uniform(50000, 100000), 0),
            },
            'trend': random.choice(['UPTREND', 'DOWNTREND', 'SIDEWAYS']),
            'relevance_score': relevance
        }
        return AgentResult(
            agent_name=self.name, confidence=relevance,
            result=result, execution_time_ms=random.uniform(50, 300)
        )

    def _calculate_relevance(self, query: str) -> float:
        query_lower = query.lower()
        matches = sum(1 for kw in self.keywords if kw.lower() in query_lower)
        return min(1.0, matches * 0.25 + 0.1)


class ScreeningAgent:
    name = 'screening'
    keywords = [
        'lọc', 'sàng lọc', 'screen', 'filter', 'so sánh', 'compare',
        'top', 'best', 'tốt nhất', 'xếp hạng', 'ranking', 'sector',
        'ngành', 'industry', 'danh sách', 'list', 'tiêu chí',
        'criteria', 'tìm', 'search', 'find', 'which', 'nào',
        'recommend', 'gợi ý', 'đề xuất', 'suggestion'
    ]

    def execute(self, query: str, context: dict = None) -> AgentResult:
        relevance = self._calculate_relevance(query)
        result = {
            'analysis_type': 'screening',
            'filtered_stocks': [
                {'ticker': 'VNM', 'score': 8.5},
                {'ticker': 'FPT', 'score': 8.2},
                {'ticker': 'VCB', 'score': 7.9},
            ],
            'criteria_used': ['P/E < 15', 'ROE > 15%', 'Market Cap > 10T'],
            'relevance_score': relevance
        }
        return AgentResult(
            agent_name=self.name, confidence=relevance,
            result=result, execution_time_ms=random.uniform(200, 800)
        )

    def _calculate_relevance(self, query: str) -> float:
        query_lower = query.lower()
        matches = sum(1 for kw in self.keywords if kw.lower() in query_lower)
        return min(1.0, matches * 0.25 + 0.1)


AGENT_REGISTRY = {
    'fundamental': FundamentalAgent(),
    'technical': TechnicalAgent(),
    'screening': ScreeningAgent(),
}
ACTION_SPACE = list(AGENT_REGISTRY.keys())
N_ACTIONS = len(ACTION_SPACE)

print(f'Agents: {ACTION_SPACE}')

## 3. Feature Extraction & Reward

In [ ]:
def extract_state(query: str) -> int:
    """
    Query text → discrete state (0-26).
    Tính relevance score từ mỗi agent, discretize thành 3 mức.
    State = score_fund*9 + score_tech*3 + score_screen
    """
    scores = []
    for agent_name in ACTION_SPACE:
        agent = AGENT_REGISTRY[agent_name]
        scores.append(agent._calculate_relevance(query))

    discretized = []
    for s in scores:
        if s < 0.3:
            discretized.append(0)   # low
        elif s < 0.6:
            discretized.append(1)   # medium
        else:
            discretized.append(2)   # high

    state = discretized[0] * 9 + discretized[1] * 3 + discretized[2]
    return state


def compute_reward(chosen_agent: str, correct_agent: str,
                   agent_result: AgentResult, difficulty: str = 'medium') -> float:
    reward = 0.0
    if chosen_agent == correct_agent:
        reward += 1.0
        if difficulty == 'hard':
            reward += 0.5
    else:
        reward -= 0.5
        if difficulty == 'easy':
            reward -= 0.3
    reward += agent_result.confidence * 0.3
    if agent_result.execution_time_ms > 500:
        reward -= 0.1
    return reward


# Demo: kiểm tra state extraction trên vài query
sample_queries = [
    ('P/E của VNM là bao nhiêu?', 'fundamental'),
    ('RSI của FPT đang ở đâu?', 'technical'),
    ('Lọc top 10 cổ phiếu ROE cao nhất', 'screening'),
]

print(f'{"Query":<50} {"State":>6} {"Correct Agent":>15}')
print('-' * 75)
for q, label in sample_queries:
    s = extract_state(q)
    scores = {name: round(AGENT_REGISTRY[name]._calculate_relevance(q), 3)
              for name in ACTION_SPACE}
    print(f'{q:<50} {s:>6}  {label:>15}  scores={scores}')

## 4. Q-Learning Orchestrator

In [ ]:
class QLearningOrchestrator:
    """
    Q-Learning Formula:
        Q(s,a) ← Q(s,a) + α[r + γ·max_a' Q(s',a') - Q(s,a)]
    """

    def __init__(self, n_states=27, n_actions=3,
                 learning_rate=0.15, discount_factor=0.9,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.99):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.q_table = np.zeros((n_states, n_actions))
        self.history = {'rewards': [], 'accuracy': [], 'epsilon': []}

    def select_action(self, state: int, training: bool = True) -> int:
        if training and random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        return int(np.argmax(self.q_table[state]))

    def update(self, state: int, action: int, reward: float, next_state: int):
        best_next = np.max(self.q_table[next_state])
        td_target = reward + self.gamma * best_next
        td_error = td_target - self.q_table[state, action]
        self.q_table[state, action] += self.lr * td_error

    def train(self, dataset: list, n_episodes: int = 200):
        print(f'Training: {n_episodes} episodes × {len(dataset)} queries')
        print(f'  LR={self.lr}, γ={self.gamma}, ε: {self.epsilon}→{self.epsilon_end}')
        print('-' * 50)

        for episode in range(n_episodes):
            random.shuffle(dataset)
            ep_reward, correct, total = 0.0, 0, 0

            for i, row in enumerate(dataset):
                query = row['query']
                correct_label = row['correct_agent']
                difficulty = row.get('difficulty', 'medium')

                state = extract_state(query)
                action_idx = self.select_action(state, training=True)
                chosen = ACTION_SPACE[action_idx]

                agent = AGENT_REGISTRY[chosen]
                result = agent.execute(query)

                reward = compute_reward(chosen, correct_label, result, difficulty)

                next_state = (extract_state(dataset[i + 1]['query'])
                              if i + 1 < len(dataset) else state)
                self.update(state, action_idx, reward, next_state)

                ep_reward += reward
                correct += int(chosen == correct_label)
                total += 1

            self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
            acc = correct / total
            self.history['rewards'].append(ep_reward)
            self.history['accuracy'].append(acc)
            self.history['epsilon'].append(self.epsilon)

            if episode % 25 == 0 or episode == n_episodes - 1:
                print(f'  Ep {episode:4d} | Reward: {ep_reward:7.2f} | '
                      f'Acc: {acc:.1%} | ε: {self.epsilon:.4f}')

        print('Training complete.')

    def evaluate(self, test_set: list) -> dict:
        rows = []
        for row in test_set:
            query = row['query']
            correct_label = row['correct_agent']
            state = extract_state(query)
            action_idx = self.select_action(state, training=False)
            chosen = ACTION_SPACE[action_idx]
            rows.append({
                'query': query[:55] + ('...' if len(query) > 55 else ''),
                'correct': correct_label,
                'chosen': chosen,
                'ok': chosen == correct_label,
                'difficulty': row.get('difficulty', '?'),
                'q_values': {ACTION_SPACE[i]: round(self.q_table[state, i], 3)
                             for i in range(self.n_actions)},
            })
        accuracy = sum(r['ok'] for r in rows) / len(rows)
        return {'accuracy': accuracy, 'n': len(rows), 'details': rows}

    def predict(self, query: str) -> dict:
        state = extract_state(query)
        action_idx = self.select_action(state, training=False)
        chosen = ACTION_SPACE[action_idx]
        result = AGENT_REGISTRY[chosen].execute(query)
        return {
            'query': query,
            'selected_agent': chosen,
            'confidence': result.confidence,
            'q_values': {ACTION_SPACE[i]: round(self.q_table[state, i], 3)
                         for i in range(self.n_actions)},
            'agent_result': result.result,
        }

print('Class QLearningOrchestrator defined.')

## 5. Chuẩn bị Dataset Train/Test từ CSV

In [ ]:
# Dùng sample 30 queries để demo nhanh
dataset = df_sample[['query', 'correct_agent', 'difficulty']].to_dict('records')

random.seed(42)
random.shuffle(dataset)
split = int(0.8 * len(dataset))
train_set = dataset[:split]
test_set = dataset[split:]

print(f'Train: {len(train_set)} | Test: {len(test_set)}')
print('Train agent dist:', {a: sum(1 for r in train_set if r["correct_agent"] == a)
                             for a in ACTION_SPACE})

## 6. Huấn Luyện

In [ ]:
orchestrator = QLearningOrchestrator(
    n_states=27, n_actions=N_ACTIONS,
    learning_rate=0.15, discount_factor=0.9,
    epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.99,
)

orchestrator.train(train_set, n_episodes=200)

## 7. Đánh Giá trên Test Set

In [ ]:
eval_result = orchestrator.evaluate(test_set)
print(f'Test Accuracy: {eval_result["accuracy"]:.1%} ({sum(r["ok"] for r in eval_result["details"])}/{eval_result["n"]})')
print()
print(f'  {"Match":<5} {"Correct":<15} {"Chosen":<15} {"Difficulty":<10} {"Query"}')
print('  ' + '-' * 90)
for r in eval_result['details']:
    mark = '✓' if r['ok'] else '✗'
    print(f'  {mark:<5} {r["correct"]:<15} {r["chosen"]:<15} {r["difficulty"]:<10} {r["query"]}')

## 8. Inference – Queries Mới

In [ ]:
new_queries = [
    'P/E ratio của VNM có hợp lý không so với ngành?',
    'RSI cho thấy FPT đang quá mua phải không?',
    'Tìm cho tôi 5 cổ phiếu ngành ngân hàng tốt nhất',
    'Phân tích xu hướng giá và đường MA của HPG',
    'Báo cáo lợi nhuận quý 4 của VCB ra sao?',
    'So sánh VNM với competitors trong ngành - mã nào nên mua?',
    'MACD của MSN có đang cho tín hiệu mua không?',
    'Gợi ý danh mục 5 mã đa dạng ngành cho 100 triệu VND',
]

print(f'  {"Query":<55} {"Agent":<15} {"Confidence"}')
print('  ' + '-' * 85)
for q in new_queries:
    pred = orchestrator.predict(q)
    print(f'  {q[:53]:<55} {pred["selected_agent"]:<15} {pred["confidence"]:.2f}')
    print(f'      Q-values: {pred["q_values"]}')

## 9. Q-Table Snapshot (Learned Policy)

In [ ]:
q = orchestrator.q_table
df_q = pd.DataFrame(q, columns=ACTION_SPACE)
df_q.index.name = 'state'
df_q['best_action'] = df_q.idxmax(axis=1)

# Chỉ hiển thị states có Q-values khác 0
df_q_nonzero = df_q[df_q[ACTION_SPACE].abs().max(axis=1) > 0.001]
print(f'Active states (non-zero Q): {len(df_q_nonzero)} / {len(df_q)}')
print()
df_q_nonzero

## 10. Training Curve (Text-based)

In [ ]:
accuracies = orchestrator.history['accuracy']
rewards = orchestrator.history['rewards']

windows = [
    ('Ep 0-50',     accuracies[:50],   rewards[:50]),
    ('Ep 50-100',   accuracies[50:100], rewards[50:100]),
    ('Ep 100-150',  accuracies[100:150], rewards[100:150]),
    ('Ep 150-200',  accuracies[150:],  rewards[150:]),
]

print(f'  {"Window":<15} {"Avg Acc":<12} {"Avg Reward"}')
print('  ' + '-' * 45)
for name, acc_w, rew_w in windows:
    if acc_w:
        print(f'  {name:<15} {np.mean(acc_w):.1%} ± {np.std(acc_w):.1%}   '
              f'{np.mean(rew_w):.2f} ± {np.std(rew_w):.2f}')

# Tìm convergence episode
threshold = 0.70
conv_ep = next((i for i, a in enumerate(accuracies) if a >= threshold), None)
if conv_ep:
    print(f'\n  Convergence {threshold:.0%}: episode {conv_ep}')
else:
    print(f'\n  Chưa đạt convergence {threshold:.0%}')